In [1]:
# CELL 1: MASTER CONFIGURATION AND HYPERPARAMETERS
# This cell controls all settings for the model, including training epochs, 
# embedding sizes, and device selection (CPU vs GPU).

import torch
import torch.nn as nn
import torch.optim as optim
import json
import re
import os
import time
import random
from torch.utils.data import Dataset, DataLoader
from torch.nn.utils.rnn import pad_sequence
from tqdm import tqdm

# --- MASTER SETTINGS ---
CONFIG = {
    "n_epochs": 1,             # Number of times to iterate over the dataset
    "batch_size": 32,           # Number of samples per training step
    "embedding_dim": 256,       # Size of word vector representations
    "hidden_dim": 512,          # Size of LSTM hidden layers
    "learning_rate": 0.001,     # Speed of optimization
    "dropout": 0.3,             # Regularization to prevent overfitting
    "teacher_forcing_ratio": 0.5,# Probability of using real labels during training
    "model_path": "atis_lstm_attn.pth", # Filename for saving the model
    "device": torch.device("cuda" if torch.cuda.is_available() else "cpu"),
    "max_sql_len": 120          # Maximum length for generated SQL queries
}

In [2]:
# CELL 2: DATA PREPROCESSING, VOCABULARY MANAGEMENT, AND SPLIT LOADING
# This cell handles the logic for tokenizing sentences, building the 
# word-to-index mappings, and filtering the JSON by "train" or "test" splits.

class Vocab:
    def __init__(self, name):
        self.name = name
        self.word2index = {"<PAD>": 0, "<SOS>": 1, "<EOS>": 2, "<UNK>": 3}
        self.index2word = {0: "<PAD>", 1: "<SOS>", 2: "<EOS>", 3: "<UNK>"}
        self.n_words = 4

    def add_sentence(self, sentence):
        for word in self.tokenize(sentence):
            if word not in self.word2index:
                self.word2index[word] = self.n_words
                self.index2word[self.n_words] = word
                self.n_words += 1

    def tokenize(self, text):
        text = text.lower().strip()
        text = re.sub(r"([,.?!\";])", r" \1 ", text) # Separate punctuation
        return text.split()

def load_split_data(file_path, split_name="train"):
    """Filter the main JSON file to only include specific question-splits."""
    with open(file_path, 'r') as f:
        data = json.load(f)
    pairs = []
    for item in data:
        sql_query = item['sql'][0] 
        for s in item['sentences']:
            if s['question-split'] == split_name:
                pairs.append((s['text'], sql_query))
    return pairs

class ATISDataset(Dataset):
    def __init__(self, pairs, input_vocab, output_vocab):
        self.pairs = pairs
        self.input_vocab = input_vocab
        self.output_vocab = output_vocab

    def __len__(self):
        return len(self.pairs)

    def sentence_to_indices(self, vocab, sentence):
        tokens = vocab.tokenize(sentence)
        indices = [vocab.word2index.get(t, vocab.word2index["<UNK>"]) for t in tokens]
        indices.append(vocab.word2index["<EOS>"])
        return torch.tensor(indices, dtype=torch.long)

    def __getitem__(self, idx):
        src_raw, trg_raw = self.pairs[idx]
        return self.sentence_to_indices(self.input_vocab, src_raw), \
               self.sentence_to_indices(self.output_vocab, trg_raw)

def collate_fn(batch):
    """Dynamic padding for batches to ensure uniform sequence lengths."""
    src_batch, trg_batch = zip(*batch)
    return pad_sequence(src_batch, batch_first=True, padding_value=0), \
           pad_sequence(trg_batch, batch_first=True, padding_value=0)

In [3]:
# CELL 3: MODEL ARCHITECTURE (LSTM + ATTENTION) AND UTILITY FUNCTIONS
# This cell defines the neural network components and the logic for 
# training steps, evaluation steps, and saving the model checkpoint.

class Encoder(nn.Module):
    def __init__(self, input_dim, emb_dim, hid_dim, dropout):
        super().__init__()
        self.embedding = nn.Embedding(input_dim, emb_dim)
        self.rnn = nn.LSTM(emb_dim, hid_dim, batch_first=True, bidirectional=True)
        self.fc = nn.Linear(hid_dim * 2, hid_dim)
        self.dropout = nn.Dropout(dropout)

    def forward(self, src):
        embedded = self.dropout(self.embedding(src))
        outputs, (hidden, cell) = self.rnn(embedded)
        # Concatenate Bi-LSTM layers and project to hidden_dim
        hidden = torch.tanh(self.fc(torch.cat((hidden[-2,:,:], hidden[-1,:,:]), dim=1)))
        cell = torch.tanh(self.fc(torch.cat((cell[-2,:,:], cell[-1,:,:]), dim=1)))
        return outputs, (hidden, cell)

class Attention(nn.Module):
    def __init__(self, hid_dim):
        super().__init__()
        self.attn = nn.Linear((hid_dim * 2) + hid_dim, hid_dim)
        self.v = nn.Linear(hid_dim, 1, bias=False)

    def forward(self, hidden, encoder_outputs):
        src_len = encoder_outputs.shape[1]
        hidden = hidden.unsqueeze(1).repeat(1, src_len, 1)
        energy = torch.tanh(self.attn(torch.cat((hidden, encoder_outputs), dim=2)))
        attention = self.v(energy).squeeze(2)
        return torch.softmax(attention, dim=1)

class AttnDecoder(nn.Module):
    def __init__(self, output_dim, emb_dim, hid_dim, dropout, attention):
        super().__init__()
        self.output_dim = output_dim
        self.attention = attention
        self.embedding = nn.Embedding(output_dim, emb_dim)
        self.rnn = nn.LSTM((hid_dim * 2) + emb_dim, hid_dim, batch_first=True)
        self.fc_out = nn.Linear((hid_dim * 2) + hid_dim + emb_dim, output_dim)
        self.dropout = nn.Dropout(dropout)

    def forward(self, input, hidden, cell, encoder_outputs):
        input = input.unsqueeze(1)
        embedded = self.dropout(self.embedding(input))
        a = self.attention(hidden, encoder_outputs).unsqueeze(1)
        weighted = torch.bmm(a, encoder_outputs)
        rnn_input = torch.cat((embedded, weighted), dim=2)
        output, (hidden, cell) = self.rnn(rnn_input, (hidden.unsqueeze(0), cell.unsqueeze(0)))
        output = torch.cat((output.squeeze(1), weighted.squeeze(1), embedded.squeeze(1)), dim=1)
        return self.fc_out(output), hidden.squeeze(0), cell.squeeze(0)

class Seq2Seq(nn.Module):
    def __init__(self, encoder, decoder, device):
        super().__init__()
        self.encoder = encoder
        self.decoder = decoder
        self.device = device

    def forward(self, src, trg, teacher_forcing_ratio=0.5):
        batch_size = src.shape[0]
        trg_len = trg.shape[1]
        outputs = torch.zeros(batch_size, trg_len, self.decoder.output_dim).to(self.device)
        encoder_outputs, (hidden, cell) = self.encoder(src)
        input = trg[:, 0]
        for t in range(1, trg_len):
            output, hidden, cell = self.decoder(input, hidden, cell, encoder_outputs)
            outputs[:, t, :] = output
            top1 = output.argmax(1)
            input = trg[:, t] if random.random() < teacher_forcing_ratio else top1
        return outputs

def save_checkpoint(model, input_vocab, output_vocab, config, loss):
    """Saves model weights, vocabs, and config for future deployment."""
    checkpoint = {
        'model_state_dict': model.state_dict(),
        'input_vocab': input_vocab,
        'output_vocab': output_vocab,
        'config': config,
        'best_loss': loss
    }
    torch.save(checkpoint, config['model_path'])

def train_epoch(loader, model, optimizer, criterion, config, epoch_idx):
    """Executes one pass through the training split."""
    model.train()
    epoch_loss = 0
    pbar = tqdm(loader, desc=f"Epoch {epoch_idx+1}/{config['n_epochs']} [Train]")
    for src, trg in pbar:
        src, trg = src.to(config['device']), trg.to(config['device'])
        optimizer.zero_grad()
        output = model(src, trg, config['teacher_forcing_ratio'])
        loss = criterion(output[:, 1:].reshape(-1, output.shape[-1]), trg[:, 1:].reshape(-1))
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()
        epoch_loss += loss.item()
        pbar.set_postfix(loss=loss.item())
    return epoch_loss / len(loader)

def evaluate_model(loader, model, criterion, config):
    """Calculates loss on the test split without using teacher forcing."""
    model.eval()
    epoch_loss = 0
    with torch.no_grad():
        for src, trg in loader:
            src, trg = src.to(config['device']), trg.to(config['device'])
            output = model(src, trg, 0) # No teacher forcing during eval
            loss = criterion(output[:, 1:].reshape(-1, output.shape[-1]), trg[:, 1:].reshape(-1))
            epoch_loss += loss.item()
    return epoch_loss / len(loader)

In [6]:
# CELL 4: EXECUTION LOOP - TRAINING ON 'TRAIN' SPLIT AND TESTING ON 'TEST' SPLIT
# This cell initializes the components, trains the model on the data 
# marked as "train", and saves the best model based on "test" split performance.

# 1. Load data according to split requirements
train_pairs = load_split_data('/Users/chillipepper/Documents/Atis Project/atis.json', split_name="train")
test_pairs = load_split_data('/Users/chillipepper/Documents/Atis Project/atis.json', split_name="test")

# 2. Build Vocabulary (Training data ONLY to prevent data leakage)
input_vocab = Vocab("English")
output_vocab = Vocab("SQL")
for src, trg in train_pairs:
    input_vocab.add_sentence(src)
    output_vocab.add_sentence(trg)

# 3. Initialize DataLoaders
train_loader = DataLoader(ATISDataset(train_pairs, input_vocab, output_vocab), 
                          batch_size=CONFIG['batch_size'], shuffle=True, collate_fn=collate_fn)
test_loader = DataLoader(ATISDataset(test_pairs, input_vocab, output_vocab), 
                         batch_size=CONFIG['batch_size'], shuffle=False, collate_fn=collate_fn)

# 4. Instantiate Seq2Seq Model
enc = Encoder(input_vocab.n_words, CONFIG['embedding_dim'], CONFIG['hidden_dim'], CONFIG['dropout'])
attn = Attention(CONFIG['hidden_dim'])
dec = AttnDecoder(output_vocab.n_words, CONFIG['embedding_dim'], CONFIG['hidden_dim'], CONFIG['dropout'], attn)
model = Seq2Seq(enc, dec, CONFIG['device']).to(CONFIG['device'])

optimizer = optim.Adam(model.parameters(), lr=CONFIG['learning_rate'])
criterion = nn.CrossEntropyLoss(ignore_index=0) # Ignore <PAD> index in loss calculation

# 5. Training and Validation Process
best_test_loss = float('inf')
print(f"Training on {len(train_pairs)} samples, Testing on {len(test_pairs)} samples.")

for epoch in range(CONFIG['n_epochs']):
    train_loss = train_epoch(train_loader, model, optimizer, criterion, CONFIG, epoch)
    test_loss = evaluate_model(test_loader, model, criterion, CONFIG)
    
    print(f"Epoch {epoch+1} Summary | Train Loss: {train_loss:.4f} | Test Loss: {test_loss:.4f}")
    
    # Save the model only if it achieves a new best loss on the test set
    if test_loss < best_test_loss:
        best_test_loss = test_loss
        save_checkpoint(model, input_vocab, output_vocab, CONFIG, best_test_loss)
        print(f"--- Checkpoint saved! Best Test Loss: {best_test_loss:.4f} ---")

Training on 4347 samples, Testing on 447 samples.


Epoch 1/1 [Train]: 100%|██████████| 136/136 [10:02<00:00,  4.43s/it, loss=0.57] 


Epoch 1 Summary | Train Loss: 1.3897 | Test Loss: 5.4540
--- Checkpoint saved! Best Test Loss: 5.4540 ---
